# HW2: symbolic regression
Alex Shields and Gopesh Baheti

The main code is in `symbolic_regression.py`. This notebook runs it on dataset1 and dataset2 and plots the results.

Settings: population 1,000, 40 generations, tournament size 4. For new generations, we select2 best, 80% crossover and 20% mutation, max depth 8.
 
Fitness is MSE divided by the variance of y. A run stops early when training error drops below 0.0001.

In [ ]:
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from symbolic_regression import run_gp, linear_scale, EARLY_STOP_THRESHOLD

SEEDS = range(212, 222)

## Data

We hold out 5,000 rows for testing (split seed 42). For dataset2 we also try dropping rows where x3 is below 0.1.

In [ ]:
def load_split(dataset, min_x3=None):
    data = np.loadtxt(f"{dataset}.csv", delimiter=",", skiprows=1)
    train, test = train_test_split(data, test_size=5000, random_state=42)
    if min_x3 is not None:
        train = train[train[:, 2] >= min_x3]
        test = test[test[:, 2] >= min_x3]
    return train[:, :-1], train[:, -1], test[:, :-1], test[:, -1]


dataset1 = load_split("dataset1")
dataset2 = load_split("dataset2")
dataset2_cut = load_split("dataset2", min_x3=0.1)

## Running the GP

Each condition runs 10 seeds. The scaled condition fits `a + b * tree` to y before scoring each tree. and Our results are saved to `results.csv`

In [ ]:
def run_condition(name, data, scale=False):
    x_train, y_train, x_test, y_test = data
    tables = []
    for seed in SEEDS:
        random.seed(seed)
        model, history, diversity = run_gp(x_train, y_train, x_train.shape[1],
                                           real_constants=(name != "Dataset1"), scale=scale)

        a, b = 0.0, 1.0
        if scale:
            a, b = linear_scale(model.evaluate(x_train), y_train)
        predicted = a + b * model.evaluate(x_test)
        test_error = np.mean((predicted - y_test) ** 2) / np.var(y_test)

        expression = str(model)
        if scale:
            expression = f"{a:.6g} + {b:.6g} * {model}"
        print(f"{name}, seed {seed}: train {history[-1]:.6g}, test {test_error:.6g}")

        run = {"kind": "run", "seed": seed, "expression": expression, "train_error": history[-1],
               "test_error": test_error, "nodes": model.size(), "generations": len(history)}
        tables.append(pd.DataFrame([run]))
        tables.append(pd.DataFrame({"kind": "history", "seed": seed, "generation": range(len(history)),
                                    "train_error": history, "unique_trees": diversity}))
        tables.append(pd.DataFrame({"kind": "prediction", "seed": seed, "x1": x_test[:, 0],
                                    "actual": y_test, "predicted": predicted}))
    return pd.concat(tables).assign(condition=name)

In [ ]:
results = pd.read_csv("results.csv", low_memory=False)
runs = results[results["kind"] == "run"]
history = results[results["kind"] == "history"]
predictions = results[results["kind"] == "prediction"]
conditions = runs["condition"].unique()

## Results


In [ ]:
runs[["condition", "seed", "train_error", "test_error", "nodes", "generations", "expression"]]

In [ ]:
runs.groupby("condition", sort=False)[["train_error", "test_error"]].median()

In [ ]:
for condition in conditions:
    best = runs[runs["condition"] == condition].sort_values("train_error").iloc[0]
    print(f"{condition} (seed {best['seed']}): {best['expression']}")


## Training error

One line per seed. The dashed line is the early stop threshold.

In [ ]:
for condition in conditions:
    fig, ax = plt.subplots(figsize=(6, 4))
    for seed in SEEDS:
        rows = history[(history["condition"] == condition) & (history["seed"] == seed)]
        ax.plot(rows["generation"], rows["train_error"])
    ax.axhline(EARLY_STOP_THRESHOLD, color="black", linestyle="--")
    ax.set(title=condition, xlabel="Generation", ylabel="Training NMSE", yscale="log")
    plt.show()

## Diversity

Number of unique trees in the population each generation, averaged over the 10 seeds.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for condition in conditions:
    rows = history[history["condition"] == condition]
    average = rows.groupby("generation")["unique_trees"].mean()
    ax.plot(average.index, average.values, label=condition)
ax.set(xlabel="Generation", ylabel="Unique trees (of 1,000)")
ax.legend()
plt.show()

## Predictions

Each plot uses the seed with the lowest training error and scored on the test rows.

In [ ]:
def best_seed(condition):
    rows = runs[runs["condition"] == condition]
    return rows.loc[rows["train_error"].idxmin(), "seed"]


seed = best_seed("Dataset1")
rows = predictions[(predictions["condition"] == "Dataset1") & (predictions["seed"] == seed)]
rows = rows.sort_values("x1")
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(rows["x1"], rows["actual"], s=5, alpha=0.4, label="Data")
ax.plot(rows["x1"], rows["predicted"], color="orange", label="GP")
ax.set(title=f"Dataset1 (seed {seed})", xlabel="x1", ylabel="y")
ax.legend()
plt.show()

In [ ]:
for condition in conditions[1:]:
    seed = best_seed(condition)
    rows = predictions[(predictions["condition"] == condition) & (predictions["seed"] == seed)]
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.scatter(rows["actual"], rows["predicted"], s=5, alpha=0.4)
    ax.plot(rows["actual"], rows["actual"], color="black", linestyle="--")
    ax.set(title=f"{condition} (seed {seed})", xlabel="Actual y", ylabel="Predicted y")
    plt.show()

## Testing for Scaling

dataset 3 has the right shape, but without scaling its predictions are about too big. Scaling only changes the size, and scales the function linearly

In [ ]:
x_train, y_train, x_test, y_test = dataset2_cut
shape_train = x_train[:, 0] * x_train[:, 1] / x_train[:, 2] ** 2
shape_test = x_test[:, 0] * x_test[:, 1] / x_test[:, 2] ** 2
a, b = linear_scale(shape_train, y_train)
print(f"a = {a:.4g}, b = {b:.4g}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(y_test, shape_test, s=5, alpha=0.4)
axes[0].plot(y_test, y_test, color="black", linestyle="--")
axes[0].set(title="Unscaled", xlabel="Actual y", ylabel="Predicted y")
axes[1].scatter(y_test, a + b * shape_test, s=5, alpha=0.4)
axes[1].plot(y_test, y_test, color="black", linestyle="--")
axes[1].set(title="Scaled", xlabel="Actual y", ylabel="Predicted y")
plt.tight_layout()
plt.show()